In [ ]:
import json
import os
from io import StringIO
from pathlib import Path

import curl_cffi
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from loguru import logger
from tqdm import tqdm

from unga81.database import get_all_countries, get_db_connection
from unga81.config import EXTERNAL_DATA_DIR, PROCESSED_DATA_DIR, FLAGS_DIR
from unga81.restcountries import download_country_data

pd.options.display.max_columns = None
pd.options.display.max_rows = None
# pd.options.display.max_rows = 20

load_dotenv()

API_KEY = os.getenv("API_KEY_RESTCOUNTRIES")

if not API_KEY:
    logger.warning(
        f"Please request a free API key in https://restcountries.com/sign-up to use the restcountries.com API"
    )

In [ ]:
conn = get_db_connection()
df = get_all_countries(conn=conn)
logger.info(f"{df.shape[0]} speeches in the database")

# RestcountriesAPI

In [ ]:
country_information_path = EXTERNAL_DATA_DIR / "restcountriesapi"
country_information_path.mkdir(parents=True, exist_ok=True)


for i, r in df.iterrows():
    if isinstance(r["iso_3"], str):
        download_country_data(alpha_3=r["iso_3"], output_path=country_information_path)

In [ ]:
# name_common = data_pais["data"]["objects"][0]["names"]["common"]
# name_official = data_pais["data"]["objects"][0]["names"]["official"]
# lat, lng = data_pais["data"]["objects"][0]["capitals"][0]["coordinates"]
# capital_name = data_pais["data"]["objects"][0]["capitals"][0]["name"]
alpha_3 = data_pais["data"]["objects"][0]["codes"]["alpha_3"]
# data_pais["data"]["objects"][0]["flag"]
alpha_3

In [ ]:
# TODO parse
# - flag
# - capital
# - area
# - timezones
# - currency
# - languages
# - borders
# - flag png
# - gini index
# - coat of arms

In [ ]:
pd.DataFrame(data_pais["data"]["objects"]).T

In [ ]:
def get_country_data(alpha_3: str):
    file_path = EXTERNAL_DATA_DIR / "restcountriesapi" / f"{alpha_3}.json"
    if not file_path.exists():
        return None

    with open(file_path, "r") as file:
        data = json.load(file)
    return data

In [ ]:
country_data = get_country_data("AGO")

In [ ]:
country_data["data"]["objects"][0]["flag"]["url_png"]
country_data["data"]["objects"][0]["area"]["kilometers"]
country_data["data"]["objects"][0]["borders"]
country_data["data"]["objects"][0]["economy"]["gini_coefficient"]
capitals = ", ".join(
    [x["name"] for x in country_data["data"]["objects"][0]["capitals"]]
)
country_data["data"]["objects"][0]["government_type"]
", ".join([x["name"] for x in country_data["data"]["objects"][0]["languages"]])
len(country_data["data"]["objects"][0]["languages"])

# Worldmeter

In [ ]:
def get_tables_from_website(url) -> list | None:
    response = requests.get(url)
    if response.ok:
        tables = pd.read_html(StringIO(response.text))
        logger.info(f"{len(tables)} tables found in {url}.")
        return tables
    return None


def parse_table(df: pd.DataFrame) -> pd.DataFrame: ...

In [ ]:
url_gdp_by_country = "https://www.worldometers.info/gdp/gdp-by-country/"
gdp_tables = get_tables_from_website(url_gdp_by_country)

In [ ]:
gdp_imf = gdp_tables[0]
gdp_wb = gdp_tables[1]

gdp_imf = gdp_imf[["Country", "GDP (Full Value, US$)", "GDP per Capita (US$)"]]
gdp_imf.columns = ["country", "gdp (us$) imf", "gdp per capita (us$) imf"]

gdp_wb = gdp_wb[["Country", "GDP (Full Value, US$)", "GDP per Capita (US$)"]]
gdp_wb.columns = ["country", "gdp (us$) wb", "gdp per capita (us$) wb"]

In [ ]:
df_gdp = gdp_imf.merge(gdp_wb, on="country")

In [ ]:
url_population = "https://www.worldometers.info/geography/countries-of-the-world/"
population_tables = get_tables_from_website(url=url_population)

In [ ]:
df_population = pd.concat([population_tables[0], population_tables[1]])
df_population = df_population[["Country", "Population 2026"]].reset_index(drop=True)
df_population.columns = ["country", "population 2026"]

In [ ]:
df_demographics = df_population.merge(df_gdp, on="country", how="left")

In [ ]:
# preprocessing data from the SDG report
country_map = {
    "Brunei": "Brunei Darussalam",
    "CÃ´te d'Ivoire": "Côte d'Ivoire",
    "Czech Republic (Czechia)": "Czechia",
    "DR Congo": "Democratic Republic of the Congo",
    "Laos": "Lao People's Democratic Republic",
    "Nauru": "Naoero",
    "North Korea": "Democratic People's Republic of Korea",
    "Russia": "Russian Federation",
    "Saint Kitts & Nevis": "Saint Kitts and Nevis",
    "Sao Tome & Principe": "Sao Tome and Principe",
    "South Korea": "Republic of Korea",
    "St. Vincent & Grenadines": "Saint Vincent & Grenadines",
    "United States": "United States of America",
    "Vietnam": "Viet Nam",
}

for i, r in df_demographics.iterrows():
    if r["country"] in country_map:
        df_demographics.at[i, "country"] = country_map[r["country"]]

df[["country", "iso_3"]].merge(df_demographics, on="country", how="left").sort_values(
    by="country"
).to_csv(PROCESSED_DATA_DIR / "worldometer.csv", index=False)

df_aux = (
    df[["country", "iso_3"]]
    .merge(df_demographics, on="country", how="outer")
    .sort_values(by="country")
)

df_aux[(df_aux["iso_3"].isnull()) | (df_aux["population 2026"].isnull())]

# SDG Report

In [ ]:
df_sdg_data = pd.read_excel(
    EXTERNAL_DATA_DIR / "sdg_index" / "database_2026.xlsx", sheet_name="SDR2026 Data"
)

In [ ]:
df_sdg_data = df_sdg_data[
    [
        "Country Code ISO3",
        "country",
        "2026 SDG Index Score",
        "2026 SDG Index Rank",
        "Goal 1 Dash",
        "Goal 1 Trend",
        "Goal 2 Dash",
        "Goal 2 Trend",
        "Goal 3 Dash",
        "Goal 3 Trend",
        "Goal 4 Dash",
        "Goal 4 Trend",
        "Goal 5 Dash",
        "Goal 5 Trend",
        "Goal 6 Dash",
        "Goal 6 Trend",
        "Goal 7 Dash",
        "Goal 7 Trend",
        "Goal 8 Dash",
        "Goal 8 Trend",
        "Goal 9 Dash",
        "Goal 9 Trend",
        "Goal 10 Dash",
        "Goal 10 Trend",
        "Goal 11 Dash",
        "Goal 11 Trend",
        "Goal 12 Dash",
        "Goal 12 Trend",
        "Goal 13 Dash",
        "Goal 13 Trend",
        "Goal 14 Dash",
        "Goal 14 Trend",
        "Goal 15 Dash",
        "Goal 15 Trend",
        "Goal 16 Dash",
        "Goal 16 Trend",
        "Goal 17 Dash",
        "Goal 17 Trend",
        "Poverty headcount ratio at $3.00/day (%)",
    ]
]

In [ ]:
df[["country", "iso_3"]].merge(
    df_sdg_data, how="left", left_on="iso_3", right_on="Country Code ISO3"
).head(10)

In [ ]:
response = requests.get("https://dashboards.sdgindex.org/profiles/")
if response.ok:
    soup = BeautifulSoup(response.text, "html.parser")

In [ ]:
links = soup.find_all("a")
data = []
base_url = "https://dashboards.sdgindex.org/profiles/"
for link in links:
    if (
        link.get("href").startswith("/profiles/")
        and len(link.get("href").replace("/profiles/", "").replace("/", "").strip()) > 0
    ):
        country_name = link.text.strip()
        country_link = link.get("href").replace("/profiles/", "")
        data.append((country_name, base_url + country_link))
df_sdg_profiles = pd.DataFrame(data, columns=["country", "sdg_profile"])

In [ ]:
df_sdg = df_sdg_data.merge(df_sdg_profiles, on="country", how="left")

In [ ]:
# df_sdg.to_csv(PROCESSED_DATA_DIR / "sdg_countries.csv", index=False)

# Other names

In [ ]:
import curl_cffi
import pandas as pd
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/List_of_alternative_country_names"
response = curl_cffi.get(url, impersonate="chrome")
soup = BeautifulSoup(response.text, "html.parser")
tables = soup.find_all(class_="wikitable")

In [ ]:
data = []
for table in tables:
    for row in table.find_all("tr")[1:]:  # ignores the column names row
        cell_data = row.find_all("td")
        if len(cell_data) == 3:  # only process when there are 3 columns
            code, name, names = cell_data
            country_name = name.find("a")["title"].strip()
            other_names = "|".join(
                [x.text.strip() for x in names.find_all("b") if len(x.text.strip()) > 2]
                + [
                    x.text.strip()
                    for x in names.find_all("i")
                    if len(x.text.strip()) > 2
                ]
            )
            data.append((code.text, country_name, other_names))

df_country_names = pd.DataFrame(data, columns=["code", "name", "alternative_names"])
df_country_names.head()

In [ ]:
df_country_names[df_country_names["name"] == "Papua New Guinea"]
df_country_names[df_country_names["name"] == "Estonia"]["alternative_names"].values[
    0
].split("|")

In [ ]:
for i, r in df_country_names.iterrows():
    for name in [x.lower().strip() for x in r["alternative_names"].split("|")]:
        if name in df[df["iso_3"] == "CHL"]["full_speech"].values[0].lower():
            print(r["name"], name)

In [ ]:
df[df["iso_3"] == "CHL"]["full_speech"].values[0].lower().replace(".", "").replace(
    ",", ""
)

# Wikipedia

In [ ]:
url_list_countries = "https://en.wikipedia.org/wiki/List_of_ISO_3166_country_codes"
response = curl_cffi.get(url_list_countries, impersonate="chrome")
response.status_code

In [ ]:
data = []
for link in soup.find("table").find_all("a"):
    if (
        "/ISO" not in link.get("href")
        and len(link.text.strip()) > 3
        and link.get("href").startswith("https://en.")
    ):
        # print(link.get("href"), link.text.strip())
        data.append((link.text.strip(), link.get("href")))

df_wikipedia_links = pd.DataFrame(data, columns=["country", "wikipedia_link"])

In [ ]:
tables_wikipedia = pd.read_html(StringIO(response.text))
tables_wikipedia[0].columns = tables_wikipedia[0].columns.droplevel(0)
df_wikipedia_codes = tables_wikipedia[0][["ISO 3166[1] name[5]", "A-2 [5]", "A-3 [5]"]]
df_wikipedia_codes.columns = ["country", "iso_2", "iso_3"]
# df_wikipedia_codes

In [ ]:
df_wikipedia = df_wikipedia_codes.merge(df_wikipedia_links, on="country")
df_wikipedia.to_csv(PROCESSED_DATA_DIR / "wikipedia_links.csv", index=False)

# Flags

In [ ]:
URL_BASE_FLAGS = "https://raw.githubusercontent.com/amckenna41/iso3166-flags/main/iso3166-1-flags/{}.svg"

# FLAGS_DIR = PROCESSED_DATA_DIR / "flags"
# FLAGS_DIR.mkdir(exist_ok=True, parents=True)
aditional_flags = {"KP": "PRK", "CZ": "CZE"}

In [ ]:
def download_flat(iso_2: str, iso_3):
    if (FLAGS_DIR / f"{iso_3}.svg").exists():
        return
    response = requests.get(URL_BASE_FLAGS.format(iso_2.lower()))
    if response.ok:
        with open(FLAGS_DIR / f"{iso_3}.svg", "w", encoding="utf-8") as f:
            f.write(response.text)


# additional flags to download
download_flat("KP", "PRK")
download_flat("CZ", "CZE")
download_flat("TR", "TUR")
download_flat("FR", "FRA")
download_flat("KR", "KOR")
download_flat("NR", "NRU")
download_flat("VN", "VNM")
download_flat("SY", "SYR")
download_flat("IR", "IRN")
download_flat("TL", "TLS")
download_flat("UA", "UKR")
download_flat("NA", "NAM")
download_flat("SZ", "SWZ")
download_flat("CI", "CIV")
download_flat("AU", "AUS")
download_flat("MK", "MKD")
download_flat("CV", "CPV")
download_flat("LA", "LAO")
download_flat("RU", "RUS")
download_flat("VA", "VAT")
download_flat("BN", "BRN")
download_flat("CG", "COG")

In [ ]:
df[df["iso_3"] == "COG"]

In [ ]:
for iso_3 in df.iso_3.to_list():
    if not (FLAGS_DIR / f"{iso_3}.svg").exists():
        print(iso_3)

In [ ]:
for i, r in tqdm(df_wikipedia.iterrows(), total=df_wikipedia.shape[0]):
    if (FLAGS_DIR / f'{r["iso_3"]}.svg').exists():
        continue
    if not isinstance(r["iso_2"], str):
        continue
    if len(r["iso_2"].lower().strip()) != 2:
        continue
    response = requests.get(URL_BASE_FLAGS.format(r["iso_2"].lower()))
    if response.ok:
        with open(FLAGS_DIR / f'{r["iso_3"]}.svg', "w", encoding="utf-8") as f:
            f.write(response.text)
    else:
        logger.warning(f'{r["iso_2"]}.svg could not be downloaded.')

In [ ]:
response = curl_cffi.get("https://en.wikipedia.org/wiki/Albania", impersonate="chrome")
response.status_code

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

In [ ]:
# Capital
# Area
# Government type
# Languages
# flag

country_card = soup.find(class_="infobox ib-country vcard")

In [ ]:
# country_card
for row in country_card.find_all("tr"):
    if "Capital" in row.text:
        print(row.find("a").text)
        capital = row.find("a").text
    if "Official" in row.text:
        print(row.find("a").text)
# country_card.find("a", title="Flag")
# country_card.find_all("td", class_="infobox-data")

In [ ]:
capital = country_card.find("th", text="Capital").find_next_sibling("td").text.strip()
languages = (
    country_card.find("th", text="Official").find_next_sibling("td").text.strip()
)
area = country_card.find("th", text="Area").find_next_sibling("td").text.strip()
government_type = (
    country_card.find("th", text="Government").find_next_sibling("td").text.strip()
)
flag = country_card.find(
    "a", href=lambda x: x and "Flag_of_Albania" in x, title="Flag of Albania"
).text.strip()

In [ ]:
for row in country_card.find_all("tr"):
    if row.text.strip().startswith("Capital"):
        # print(row.get('title'))
        print(row.find("td").find("a").text)
    if row.text.strip().startswith("Official"):
        print(row)

    # else:
    #     print(row.text)

In [ ]:
df_country_card = pd.read_html(StringIO(response.text))[0]
df_country_card.columns = ["key", "value"]
df_country_card

In [ ]:
for i, r in df_country_card.iterrows():
    # if str(r["key"]).strip() == "Official languages":
    if r["key"] == "Government":
        print("*" * 20)
        print(f'{r["key"]} - {r["value"]}')
        print("*" * 20)
    if "Capital" in str(r["key"]):
        print("*" * 20)
        print(f'{r["key"]} - {r["value"]}')
        print("*" * 20)
    else:
        print(f'{r["key"]} - {r["value"]}')